# 第1模块，第1节：基础概念

在本节中，我们将为虚拟的电子产品电商商店TechHub构建一个简单的客服系统。

<div align="center">
    <img src="../../images/db_agent.png">
</div>

我们从基础开始：
1. 使用消息进行LLM调用的方法
2. 定义能够访问外部数据（如TechHub数据库）的工具
3. 手动工具调用循环（虽然繁琐，但非常教育意义）

到此结束时，您将理解为什么代理框架存在——提示：它们自动化的部分正是我们在后面要手动实现的部分。

# 预备工作

首先，我们加载我们的环境变量（API密钥）。

In [5]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

False

# 1. 基本 LLM 调用

让我们从简单开始——仅使用一条消息调用一个 LLM。

**注意：** 通过 .env 文件中的 WORKSHOP_MODEL 设置默认模型（参见 config.py）。

In [6]:
from langchain.chat_models import init_chat_model
from config import DEFAULT_MODEL

print(f"Using model: {DEFAULT_MODEL}")

# Initialize the model (using workshop default)
llm = init_chat_model(DEFAULT_MODEL)

# Simple string input
response = llm.invoke("What is LangChain in under 10 words?")
response.pretty_print()

Using model: anthropic:claude-haiku-4-5


TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

## 2. 使用消息

LLMs 与 **消息** 进行交互，这些消息具有不同的角色：
- `SystemMessage`：指示 LLM 应该采取的行为指南
- `HumanMessage`：用户的输入
- `AIMessage`：模型的响应
- `ToolMessage`：工具执行的结果（我们很快就会看到这一点！）

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Multi-turn conversation with messages
messages = [
    SystemMessage(
        content="You are a helpful customer support assistant for TechHub, an electronics store."
    ),
    HumanMessage(content="Hello!"),
    AIMessage(content="Hi! How can I help you today?"),
    HumanMessage(content="What's the status of order ORD-2024-0123?"),
]

response = llm.invoke(messages)
response.pretty_print()

**注意问题：** LLM无法真正查看订单！它没有访问我们的数据库的权限。

这是工具发挥作用的地方。

## 3. 定义工具

<div align="center">
    <img src="../../images/db_tools.png">
</div>

工具使LLM能够与外部系统交互。让我们创建四个用于查询我们[TechHub数据库](../../data/structured/techhub_schema.png)的工具。

`@tool` 装饰器从 `langchain` 自动执行以下操作：
- 提取LLM的函数签名
- 使用文档说明作为工具描述
- 处理输入输出格式

In [ ]:
from pathlib import Path
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

# instaniate the database connection
DB_PATH = Path("../../data/structured/techhub.db")
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")


# helper function
def extract_values(result):
    """Convert SQLDatabase query results (list of dicts) to list of tuples (values only)."""
    return [tuple(row.values()) for row in result]


@tool
def get_order_status(order_id: str) -> str:
    """Get status, dates, and tracking information for a specific order.

    Note: For order total, calculate from items using get_order_item_price().

    Args:
        order_id: The order ID (e.g., "ORD-2024-0123")

    Returns:
        Formatted string with order status, dates, and tracking number.
    """
    result = db._execute(
        f"""
        SELECT order_id, order_date, status, shipped_date, tracking_number
        FROM orders
        WHERE order_id = '{order_id}'
    """
    )
    result = extract_values(result)

    if not result:
        return f"Order {order_id} not found."

    order_id, order_date, status, shipped_date, tracking_number = result[0]

    response = f"Order {order_id}:\n"
    response += f"- Status: {status}\n"
    response += f"- Order Date: {order_date}\n"

    if shipped_date:
        response += f"- Shipped Date: {shipped_date}\n"
    if tracking_number:
        response += f"- Tracking Number: {tracking_number}\n"

    return response


@tool
def get_order_items(order_id: str) -> str:
    """Get list of items in a specific order with product IDs and quantities.

    Note: For pricing, use get_order_item_price() for historical price paid, or get_product_info() for current price.

    Args:
        order_id: The order ID (e.g., "ORD-2024-0123")

    Returns:
        Formatted string with product IDs and quantities (no prices).
    """
    result = db._execute(
        f"""
        SELECT product_id, quantity
        FROM order_items
        WHERE order_id = '{order_id}'
    """
    )
    result = extract_values(result)

    if not result:
        return f"No items found for order {order_id}."

    response = f"Items in order {order_id}:\n"
    for product_id, quantity in result:
        response += f"- Product ID: {product_id}, Quantity: {quantity}\n"

    return response


@tool
def get_product_info(product_identifier: str) -> str:
    """Get product details by product name or product ID.

    Args:
        product_identifier: Product name (e.g., "MacBook Air") or product ID (e.g., "TECH-LAP-001")

    Returns:
        Formatted string with product name, category, price, and stock status.
    """
    # Try exact ID match first
    result = db._execute(
        f"""
        SELECT product_id, name, category, price, in_stock
        FROM products
        WHERE product_id = '{product_identifier}'
    """
    )
    result = extract_values(result)

    # If no exact match, try fuzzy name search
    if not result:
        result = db._execute(
            f"""
            SELECT product_id, name, category, price, in_stock
            FROM products
            WHERE name LIKE '%{product_identifier}%'
            LIMIT 1
        """
        )
        result = extract_values(result)

    if not result:
        return f"Product '{product_identifier}' not found."

    product_id, name, category, price, in_stock = result[0]
    stock_status = "In Stock" if in_stock else "Out of Stock"

    return f"{name} ({product_id})\n- Category: {category}\n- Price: ${price:.2f}\n- Status: {stock_status}"


@tool
def get_order_item_price(order_id: str, product_id: str) -> str:
    """Get the historical price paid for a specific item in an order.

    Use this to get the actual price the customer paid at time of purchase,
    which may differ from the current retail price from get_product_info().

    Args:
        order_id: The order ID (e.g., "ORD-2024-0123")
        product_id: The product ID (e.g., "TECH-LAP-001")

    Returns:
        Formatted string with historical price per unit.
    """
    result = db._execute(
        f"""
        SELECT price_per_unit, quantity
        FROM order_items
        WHERE order_id = '{order_id}' AND product_id = '{product_id}'
        """
    )
    result = extract_values(result)

    if not result:
        return f"Item {product_id} not found in order {order_id}."

    price, quantity = result[0]
    return f"Historical price for {product_id} in {order_id}: ${price:.2f} per unit (quantity: {quantity})"

让我们测试一下 `get_order_status` Tool。

In [ ]:
example_order = "ORD-2024-0127"
result = get_order_status.invoke(example_order)
print(result)

让我们查看如何通过`@tool`装饰器解析工具信息并将其解析为一个`StructuredTool`对象。

In [ ]:
print("\n--- TOOL NAME ---")
print(get_order_status.name)

print("\n--- TOOL DESCRIPTION ---")
print(get_order_status.description)

print("\n--- TOOL ARGUMENTS ---")
print(get_order_status.args)

In [ ]:
get_order_status.tool_call_schema.model_json_schema()

## 4. 实际工具调用的LLM 使用流程

现在让我们看看LLMs如何实际[使用工具](https://docs.langchain.com/oss/python/langchain/models#tool-calling)！这个过程分为几个阶段：

<div align="center">
    <img src="../../images/function-calling.png">
    <br>
    <sub>
        图片来源：<a href="https://www.philschmid.de/gemini-function-calling">Phil Schmid 博客</a>
    </sub>
</div>

让我们一步步看这个流程是如何运作的！

### 步骤0：将工具绑定到模型

首先，我们将LLM告诉它可用的工具有哪些，这称为“绑定”工具。这样可以确保与模型请求一起包含所有可用工具的定义。

In [ ]:
# Bind tools to the model - this tells the LLM what tools are available
tools = [get_order_status, get_order_items, get_product_info, get_order_item_price]
llm_with_tools = llm.bind_tools(tools)

### 步骤 1, 2, 3: 调用LLM进行查询，它决定调用一个工具，并返回格式化的工具调用对象

现在让我们给LLM发送一个查询。它将根据可用的工具描述决定调用哪个工具，并返回包含工具参数的工具调用对象。

In [ ]:
# Ask about an order
messages = [
    SystemMessage(content="You are a helpful customer support assistant."),
    HumanMessage(content="What's the status of order ORD-2024-0123?"),
]

# Call the LLM
response = llm_with_tools.invoke(messages)
response.pretty_print()

# Add the AI's tool call to messages
messages.append(response)

### 第4步：手动执行工具

LLM告诉我们_什么需要调用_以及_带有哪些参数_. 现在**我们**需要_实际执行工具_，按照指定的参数进行。

In [ ]:
# Extract the tool call information
tool_call = response.tool_calls[0]
tool_name = tool_call["name"]
tool_args = tool_call["args"]
tool_call_id = tool_call["id"]

print(f"[Executing]\n Tool: {tool_name}\n Args: {tool_args}\n ID: {tool_call_id}", "\n")

# Manually execute the tool
if tool_name == "get_order_status":
    tool_result = get_order_status.invoke(tool_args)
elif tool_name == "get_order_items":
    tool_result = get_order_items.invoke(tool_args)
elif tool_name == "get_product_info":
    tool_result = get_product_info.invoke(tool_args)
elif tool_name == "get_order_item_price":
    tool_result = get_order_item_price.invoke(tool_args)

print(f"[Tool Result]\n {tool_result}")

### 第五步：将结果反馈给LLM

现在我们需要将工具的结果反馈给LLM，以便它能够生成一个用户自然语言中最终的回答。请注意，在将工具结果添加到消息历史记录时，我们使用了`ToolMessage`类型。

In [ ]:
from langchain_core.messages import ToolMessage

# Create a ToolMessage with the result
tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"],  # Must match the ID from the tool call
)
messages.append(tool_message)
messages

### 第六步：获取最终答案

现在LLM已经得到了工具的结果，并能够用自然语言为用户提供一个完整的回答。

In [ ]:
# Call the LLM again with the tool result
final_response = llm_with_tools.invoke(messages)

final_response.pretty_print()

### 综合起来：循环调用

在实际场景中，LLM可能需要调用多个工具或进行多次循环。让我们创建一个函数来自动化这个过程:

In [ ]:
def run_agent_loop(user_query: str):
    """Run the complete tool calling loop."""

    messages = [
        SystemMessage(
            content="You are a helpful customer support assistant for TechHub."
        ),
        HumanMessage(content=user_query),
    ]

    print(f"\n{'='*60}")
    print(f"User: {user_query}")
    print(f"{'='*60}\n")

    # Keep looping until we get a final answer
    iteration = 0
    while True:
        iteration += 1

        # Call the LLM
        response = llm_with_tools.invoke(messages)

        # Check if LLM wants to use tools
        if not response.tool_calls:
            # No tools needed - we have the final answer!
            print(f"[Final Answer]\n {response.content}\n")
            break

        # LLM wants to use tools
        print(
            f"[Iteration {iteration}] LLM is calling {len(response.tool_calls)} tool(s)..."
        )
        messages.append(response)

        # Execute each tool
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            # Execute the tool
            if tool_name == "get_order_status":
                result = get_order_status.invoke(tool_args)
            elif tool_name == "get_order_items":
                result = get_order_items.invoke(tool_args)
            elif tool_name == "get_product_info":
                result = get_product_info.invoke(tool_args)
            elif tool_name == "get_order_item_price":
                result = get_order_item_price.invoke(tool_args)

            arg_str = ", ".join(f"{k}={v!r}" for k, v in tool_args.items())
            print(f"  → {tool_name}({arg_str})")

            # Add tool result to messages
            messages.append(
                ToolMessage(content=str(result), tool_call_id=tool_call["id"])
            )

        print()  # Blank line before next iteration

### 尝试一下！

现在，让我们用不同的查询测试我们的代理循环：

**示例 1：简单的订单查询**


In [ ]:
run_agent_loop("What's the status of order ORD-2024-0123?")

**示例 2：商品价格查询**


In [ ]:
run_agent_loop("How much does the MacBook Air cost?")

**示例3：一个查询中使用多个工具**

In [ ]:
run_agent_loop(
    "What's the status of order ORD-2024-0123, what was in it, and how much did it cost?"
)

> **提示:** 想知道代理框架运行这个循环背后发生了什么吗？查看**[bonus_react_loop_langgraph_primitives.ipynb](bonus_react_loop_langgraph_primitives_zh_CN.ipynb)**，使用LangGraph的`StateGraph`、节点和边直接构建相同的代理——这就是创建代理所用到的基本元素，也是第3节和4节中多代理模式的基础。